# Phase 2C.4.1: Attack Ground Truth and Process Telemetry Temporal Mapping

**Status:** READ-ONLY AUDIT & TEMPORAL MAPPING  
**Primary Database:** `dataset/merged_datasets.duckdb`  
**Impact Ground Truth:** `dataset/impact_assessment.duckdb`  

### Strict Constraints:
- Read-only audit only.
- Zero model evaluation (no ROC-AUC, PR-AUC, F1, Precision, Recall).
- Zero threshold tuning on attack data.
- Baseline LSTM Autoencoder model weights and architecture remain frozen.
- Zero attack data fed to the ML training or scaling pipelines.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import duckdb
import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

DB_PATH = project_root / "dataset" / "merged_datasets.duckdb"
IMPACT_PATH = project_root / "dataset" / "impact_assessment.duckdb"

con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute(f"ATTACH '{IMPACT_PATH}' AS impact (READ_ONLY)")
print("DuckDB connected in read-only mode.")

## 1. Inspect Experimental Runs Catalog
Inspect all 6 experimental runs registered in the dataset catalog.

In [ ]:
runs_df = con.execute("SELECT id, relative_path FROM dataset ORDER BY relative_path").fetchdf()
runs_df["run_name"] = runs_df["relative_path"].apply(lambda x: x.split('/')[-1])
runs_df

## 2. Process Telemetry Sampling and Monotonicity
Verify sampling interval, monotonicity, and temporal continuity across runs.

In [ ]:
proc_stats = []
for _, r in runs_df.iterrows():
    ds_id = str(r['id'])
    pv_ts = con.execute(f"SELECT ts FROM pv_process_data WHERE dataset_id = '{ds_id}' ORDER BY ts").fetchdf()['ts']
    if len(pv_ts) > 0:
        dt_series = pv_ts.diff().dt.total_seconds().dropna()
        proc_stats.append({
            "run_name": r["run_name"],
            "rows": len(pv_ts),
            "earliest_ts": str(pv_ts.min()),
            "latest_ts": str(pv_ts.max()),
            "is_monotonic": pv_ts.is_monotonic_increasing,
            "median_dt_s": round(float(dt_series.median()), 4),
            "mean_dt_s": round(float(dt_series.mean()), 4),
            "seq_60_duration_s": round(float(60 * dt_series.median()), 2)
        })

df_proc_stats = pd.DataFrame(proc_stats)
df_proc_stats

## 3. Attack Ground Truth: Sessions and Execution Steps
Extract discrete start/stop intervals from `exec_steps` and check coverage against process telemetry.

In [ ]:
alignment_summary = []
for _, r in runs_df[~runs_df["run_name"].str.contains("normal")].iterrows():
    ds_id = str(r['id'])
    run_name = r['run_name']
    
    pv_ts = con.execute(f"SELECT ts FROM pv_process_data WHERE dataset_id = '{ds_id}' ORDER BY ts").fetchdf()['ts']
    pv_start, pv_end = pv_ts.min(), pv_ts.max()
    
    steps_df = con.execute(f"""
        SELECT 
            es.attack_session_id, es.step_idx, es.worker_id,
            MIN(CASE WHEN es.kind = 'start' THEN es.exec_ts END) as start_ts,
            MAX(CASE WHEN es.kind = 'stop' THEN es.exec_ts END) as stop_ts
        FROM exec_steps es
        JOIN attack_session a ON es.attack_session_id = a.id
        WHERE a.dataset_id = '{ds_id}'
        GROUP BY 1, 2, 3
    """).fetchdf()
    
    valid_steps = steps_df.dropna(subset=['start_ts', 'stop_ts'])
    overlaps = (valid_steps['stop_ts'] >= pv_start) & (valid_steps['start_ts'] <= pv_end)
    lead_in_s = (valid_steps['start_ts'].min() - pv_start).total_seconds()
    lead_out_s = (pv_end - valid_steps['stop_ts'].max()).total_seconds()
    durations = (valid_steps['stop_ts'] - valid_steps['start_ts']).dt.total_seconds()
    
    alignment_summary.append({
        "run_name": run_name,
        "total_steps": len(steps_df),
        "valid_steps": len(valid_steps),
        "overlapping_steps": int(overlaps.sum()),
        "overlap_pct": round(float(overlaps.mean() * 100), 2),
        "clean_lead_in_min": round(lead_in_s / 60, 2),
        "clean_lead_out_min": round(lead_out_s / 60, 2),
        "median_step_duration_s": round(float(durations.median()), 1)
    })

df_alignment = pd.DataFrame(alignment_summary)
df_alignment

## 4. Sequence Labeling Feasibility & Strategy Comparison
Simulate candidate 60-step sequence labeling strategies across all 5 attack runs.

In [ ]:
strat_rows = []
for _, r in runs_df[~runs_df["run_name"].str.contains("normal")].iterrows():
    ds_id = str(r['id'])
    run_name = r['run_name']
    
    ts_series = con.execute(f"SELECT ts FROM pv_process_data WHERE dataset_id = '{ds_id}' ORDER BY ts").fetchdf()['ts']
    n_samples = len(ts_series)
    n_seq = n_samples - 60 + 1
    ts_vals = ts_series.values
    
    steps_df = con.execute(f"""
        SELECT 
            es.attack_session_id, es.step_idx, es.worker_id,
            MIN(CASE WHEN es.kind = 'start' THEN es.exec_ts END) as start_ts,
            MAX(CASE WHEN es.kind = 'stop' THEN es.exec_ts END) as stop_ts
        FROM exec_steps es
        JOIN attack_session a ON es.attack_session_id = a.id
        WHERE a.dataset_id = '{ds_id}'
        GROUP BY 1, 2, 3
    """).fetchdf().dropna(subset=['start_ts', 'stop_ts'])
    
    starts = steps_df['start_ts'].values
    stops = steps_df['stop_ts'].values
    
    point_mask = np.zeros(n_samples, dtype=bool)
    for s, e in zip(starts, stops):
        mask = (ts_vals >= s) & (ts_vals <= e)
        point_mask[mask] = True
        
    windowed = sliding_window_view(point_mask, window_shape=60)
    step_counts = windowed.sum(axis=1)
    
    strat_a = step_counts >= 1
    strat_b = point_mask[30 : 30 + n_seq]
    strat_c_50 = step_counts >= 30
    strat_d = point_mask[59 : 59 + n_seq]
    
    strat_rows.append({
        "run_name": run_name,
        "sequences": n_seq,
        "point_attack_pct": round(float(point_mask.mean() * 100), 2),
        "strat_A_any_overlap_pct": round(float(strat_a.mean() * 100), 2),
        "strat_B_midpoint_pct": round(float(strat_b.mean() * 100), 2),
        "strat_C_50pct_overlap_pct": round(float(strat_c_50.mean() * 100), 2),
        "strat_D_causal_end_pct": round(float(strat_d.mean() * 100), 2)
    })

df_strat_comparison = pd.DataFrame(strat_rows)
df_strat_comparison

## 5. Physical Ground Truth: Subsystem Targeting & Impact Rate
Inspect `judge_results` to understand how many attack steps targeted Solar/Wind vs. Battery, and how many caused observable physical deltas.

In [ ]:
subsystem_df = con.execute("""
    SELECT 
        j.connection_table,
        count(*) as total_steps,
        sum(case when j.delta_observed_during_active then 1 else 0 end) as impactful_steps,
        round(sum(case when j.delta_observed_during_active then 1 else 0 end) * 100.0 / count(*), 2) as impact_pct
    FROM impact.judge_results j
    GROUP BY 1
    ORDER BY total_steps DESC
""").fetchdf()
subsystem_df

## 6. Audit Conclusions
- **Primary Ground Truth Table:** `exec_steps` provides discrete start and stop timestamps for actual physical execution intervals.
- **Sequence Duration:** Exactly 60 steps $\times 0.5254\text{s} = 31.52\text{ seconds}$.
- **Temporal Containment:** 100.0% of valid attack steps fall completely within process telemetry recordings.
- **Recommended Evaluation Strategy:** Strategy D (causal sequence end $t_{59}$) for real-time monitoring benchmarks, with sensitivity analysis across Strategy A and C1.